# BSDE Pricing via LSMC
## Probabilists' Hermite Basis — European · Basket · American

---

### Structure

| Section | Instrument | Benchmark |
|---------|-----------|-----------|
| 1 | 1-D European call | Black–Scholes |
| 2 | Multi-dim basket call (3-D, 5-D), heterogeneous $\sigma_j$, $S_0^{(j)}$ | Antithetic MC |
| 3 | 1-D American put (Reflected BSDE) | CRR binomial tree |

All sections verify **price** and **delta** (with Monte Carlo confidence intervals) and show convergence plots.

---

## BSDE Framework

The FBSDE pair $(X,Y,Z)$ on $[0,T]$:

$$
\begin{cases}
dX_t = b(t,X_t)\,dt + \sigma(t,X_t)\,dW_t^{\text{corr}}, \quad X_0=x_0,\\
Y_t = \Phi(X_T) + \displaystyle\int_t^T f(s,X_s,Y_s,Z_s)\,ds - \int_t^T Z_s^\top dW_s^{\text{ind}}.
\end{cases}
$$

**Key convention (following GLW 2005):** $Z$ is defined w.r.t. **independent** Brownian motions $W^{\text{ind}}$. Asset paths use correlated increments $\Delta W^{\text{corr}} = L\,\Delta W^{\text{ind}}$ where $LL^\top = \rho$. This choice ensures the $Z$ regression is orthogonal and requires no correlation correction.

Under the risk-neutral measure, the pricing BSDE has driver $f(t,y,z)=-ry$, giving the DP equations:

$$
\boxed{
Y_{t_n} = \Phi(X_{t_n}),\qquad
Z_{t_i} = \frac{1}{\Delta t}\mathbb{E}\!\left[Y_{t_{i+1}}\,\Delta W_i^{\text{ind}}\;\middle|\;X_{t_i}\right],\qquad
Y_{t_i} = (1-r\Delta t)\,\mathbb{E}\!\left[Y_{t_{i+1}}\;\middle|\;X_{t_i}\right].
}
$$

---

## Why Probabilists' Hermite Polynomials

$He_n$ are orthogonal w.r.t. $\mathcal{N}(0,1)$:

$$
He_n(x)=(-1)^n e^{x^2/2}\tfrac{d^n}{dx^n}e^{-x^2/2},\qquad
\int He_m\,He_n\,\tfrac{e^{-x^2/2}}{\sqrt{2\pi}}dx = n!\,\delta_{mn}.
$$

Since $\log S_{t_i}$ is Gaussian under GBM, standardised log-prices give near-orthogonal regression features. `scipy.special.hermitenorm(n)` implements $He_n$; the physicists' `hermite(n)` uses a different normalisation and is **not** used here.


## 0. Imports

In [ ]:
import numpy as np
import pandas as pd

from scipy.stats import norm
from scipy import special
import statsmodels.api as sm

import plotly.graph_objects as go

import warnings
warnings.filterwarnings('ignore')

SEED = 123   # for reproducibility

## 1. Market Parameters

`MarketParams1D` for scalar instruments, `BasketParams` for multi-asset.
All solvers accept these objects — sweep parameters without touching solver logic.

In [2]:
from dataclasses import dataclass

@dataclass
class MarketParams1D:
    S0    : float = 100.0
    K     : float = 100.0
    r     : float = 0.05
    sigma : float = 0.20
    T     : float = 1.0

@dataclass
class BasketParams:
    S0_vec    : np.ndarray = None   # shape (d,)
    K         : float      = 100.0
    r         : float      = 0.05
    sigma_vec : np.ndarray = None   # shape (d,)
    rho       : float      = 0.40
    T         : float      = 1.0

    def corr(self):
        d = len(self.S0_vec)
        return self.rho * np.ones((d, d)) + (1 - self.rho) * np.eye(d)

    def cholesky(self):
        return np.linalg.cholesky(self.corr())

## 2. GBM Simulation

$$
\Delta W^{\text{ind}} = \sqrt{\Delta t}\,Z,\quad Z\sim\mathcal{N}(0,I_d),\qquad
\Delta W^{\text{corr}} = L\,\Delta W^{\text{ind}},\quad LL^\top=\rho.
$$


In [3]:
def simulate_gbm_1d(p: MarketParams1D, n: int, N: int):
    dt = p.T / n
    dW = np.sqrt(dt) * np.random.randn(n, N)
    W  = np.vstack([np.zeros((1, N)), np.cumsum(dW, axis=0)])
    t_ = np.linspace(0, p.T, n + 1).reshape(-1, 1)
    S  = p.S0 * np.exp(p.sigma * W + (p.r - 0.5 * p.sigma**2) * t_)
    return S, dW          # (n+1,N), (n,N)


def simulate_gbm_basket(bp: BasketParams, n: int, N: int):
    d        = len(bp.S0_vec)
    dt       = bp.T / n
    L        = bp.cholesky()
    dW_ind   = np.sqrt(dt) * np.random.randn(n, N, d)
    dW_corr  = dW_ind @ L.T
    log_S    = np.zeros((n + 1, N, d))
    log_S[0] = np.log(bp.S0_vec)
    drift    = (bp.r - 0.5 * bp.sigma_vec**2) * dt
    for i in range(n):
        log_S[i + 1] = log_S[i] + drift + bp.sigma_vec * dW_corr[i]
    return np.exp(log_S), dW_ind, dW_corr  # (n+1,N,d), (n,N,d), (n,N,d)

## 3. Basis Functions

**1-D (Y regressions everywhere):** Hermite on standardised $\log(S/K)$.

**Multi-D (Z regressions in basket):** Full degree-2 polynomial in $\log(S_j/S_0^{(j)})$, including cross-terms. Basis size $= 1 + d + d + \binom{d}{2}$; well-conditioned and avoids the $\binom{d+p}{p}$ explosion.


In [4]:
def hermite_basis_1d(x: np.ndarray, degree: int) -> np.ndarray:
    xs = (x - x.mean()) / (x.std() + 1e-12)
    return np.column_stack([special.hermitenorm(k)(xs) for k in range(degree + 1)])


def poly_basis_nd(log_s_rel: np.ndarray, degree: int = 2) -> np.ndarray:
    N, d = log_s_rel.shape
    cols = [np.ones(N)]
    for j in range(d): cols.append(log_s_rel[:, j])
    if degree >= 2:
        for j in range(d): cols.append(log_s_rel[:, j] ** 2)
        for j in range(d):
            for k in range(j + 1, d): cols.append(log_s_rel[:, j] * log_s_rel[:, k])
    return np.column_stack(cols)


def ols_predict(Phi: np.ndarray, target: np.ndarray) -> np.ndarray:
    return Phi @ sm.OLS(target, Phi).fit().params

---
## Section 1 — 1-D European Call

### BSDE

$$
-dV_t = -rV_t\,dt - Z_t\,dW_t^{\text{ind}}, \qquad V_T=(S_T-K)^+.
$$

**DP equations** ($\text{disc}=1-r\Delta t$):

$$
Y_{t_n}=(S_{t_n}-K)^+,\qquad
Y_{t_i}=\text{disc}\cdot\hat{\mathbb{E}}[Y_{t_{i+1}}\mid S_{t_i}],\qquad
Z_{t_i}=\frac{1}{\Delta t}\hat{\mathbb{E}}[Y_{t_{i+1}}\,\Delta W_i^{\text{ind}}\mid S_{t_i}].
$$

`disc` multiplies outside the regression target — regress raw $Y_{i+1}$, multiply fitted values by `disc` (GLW 2005, eq. 2.2).

### Delta

$$
Z_0 = \mathbb{E}[Y_1\,\Delta W_0^{\text{ind}}]/\Delta t,\qquad \Delta_0 = Z_0/(\sigma S_0).
$$


### Methodology Note

Results are based on =10$ independent runs of =50\,000$ paths each, identical in structure to the American put (Section~3) and basket (Section~2) experiments. The BSDE-LSMC mean $\pm 2\hat\sigma$ band is the empirical 95\% confidence interval across runs. A bias-to-\hat\sigma$ ratio below 1 means the deviation from the benchmark is indistinguishable from Monte Carlo noise.


In [5]:
def lsmc_bsde_european(p: MarketParams1D, n: int, N: int, degree: int = 4):
    dt   = p.T / n
    disc = 1.0 - p.r * dt
    S, dW = simulate_gbm_1d(p, n, N)
    Y = np.zeros((n + 1, N))
    Z = np.zeros((n + 1, N))
    Y[n] = np.maximum(S[n] - p.K, 0.0)
    for i in range(n - 1, 0, -1):
        B    = hermite_basis_1d(np.log(S[i] / p.K), degree)
        Y[i] = disc * ols_predict(B, Y[i + 1])
        Z[i] = ols_predict(B, Y[i + 1] * dW[i]) / dt
    B0   = hermite_basis_1d(np.log(S[1] / p.K), degree)
    Z[0] = ols_predict(B0, Y[1] * dW[0]) / dt
    price = disc * Y[1].mean()
    delta = Z[0].mean() / (p.sigma * p.S0)
    return price, delta, Y, Z


def bs_call(p: MarketParams1D):
    d1    = (np.log(p.S0 / p.K) + (p.r + 0.5 * p.sigma**2) * p.T) / (p.sigma * np.sqrt(p.T))
    d2    = d1 - p.sigma * np.sqrt(p.T)
    price = p.S0 * norm.cdf(d1) - p.K * np.exp(-p.r * p.T) * norm.cdf(d2)
    return float(price), float(norm.cdf(d1)), float(p.sigma * p.S0 * norm.cdf(d1))

In [6]:
p1 = MarketParams1D(S0=100., K=100., r=0.05, sigma=0.20, T=1.0)

bs_price, bs_delta, bs_z0 = bs_call(p1)

# R independent runs — consistent with American and basket sections
R_EUR = 10
N_EUR = 50_000
np.random.seed(SEED)
eur_prices, eur_deltas, eur_z0s = [], [], []
for _ in range(R_EUR):
    p_, d_, Y_eur, Z_eur = lsmc_bsde_european(p1, n=50, N=N_EUR, degree=4)
    eur_prices.append(p_)
    eur_deltas.append(d_)
    eur_z0s.append(Z_eur[0].mean())

eur_price_mean = np.mean(eur_prices);  eur_price_std = np.std(eur_prices)
eur_delta_mean = np.mean(eur_deltas);  eur_delta_std = np.std(eur_deltas)
eur_z0_mean    = np.mean(eur_z0s);     eur_z0_std    = np.std(eur_z0s)

print(f"{'Method':<30} {'Price':>10}  {'Delta':>9}  {'Z0':>10}")
print("-" * 65)
print(f"{'Black-Scholes (exact)':<30} {bs_price:>10.6f}  {bs_delta:>9.6f}  {bs_z0:>10.6f}")
print(f"{'BSDE-LSMC mean':<30} {eur_price_mean:>10.6f}  {eur_delta_mean:>9.6f}  {eur_z0_mean:>10.6f}")
print(f"{'BSDE-LSMC 2-sigma CI':<30} {2*eur_price_std:>10.6f}  {2*eur_delta_std:>9.6f}  {2*eur_z0_std:>10.6f}")
print("-" * 65)
print(f"{'Bias (mean - BS)':<30} {eur_price_mean-bs_price:>+10.6f}  {eur_delta_mean-bs_delta:>+9.6f}  {eur_z0_mean-bs_z0:>+10.6f}")
print(f"{'Relative bias':<30} {(eur_price_mean-bs_price)/bs_price:>+9.4%}  {(eur_delta_mean-bs_delta)/bs_delta:>+8.4%}  {(eur_z0_mean-bs_z0)/bs_z0:>+9.4%}")
print(f"{'Bias / 2-sigma':<30} {abs(eur_price_mean-bs_price)/(2*eur_price_std):>10.2f}x  {abs(eur_delta_mean-bs_delta)/(2*eur_delta_std):>9.2f}x")
print()
print(f"R={R_EUR} independent runs x N={N_EUR:,} paths each.")
print("'Bias / 2-sigma' < 1 means deviation is within noise — no structural error.")

Method                              Price      Delta          Z0
-----------------------------------------------------------------
Black-Scholes (exact)           10.450584   0.636831   12.736613
BSDE-LSMC mean                  10.446922   0.637404   12.748072
BSDE-LSMC 2-sigma CI             0.149642   0.048618    0.972354
-----------------------------------------------------------------
Bias (mean - BS)                -0.003662  +0.000573   +0.011459
Relative bias                   -0.0350%  +0.0900%   +0.0900%
Bias / 2-sigma                       0.02x       0.01x

R=10 independent runs x N=50,000 paths each.
'Bias / 2-sigma' < 1 means deviation is within noise — no structural error.


### Convergence in $N$

Total error: $|\hat V_0 - V_0|^2 \lesssim C_1\Delta t + C_2\varepsilon_\text{proj}^2 + C_3/N$ (GLW 2005). Fixing $n=50$ and $p=4$, varying $N$ isolates the $\mathcal{O}(N^{-1/2})$ Monte Carlo term.


In [7]:
degrees_eur = [1, 3, 4]
Ns_eur      = list(range(5_000, 55_000, 5_000))

np.random.seed(SEED)
conv_eur = {d: [] for d in degrees_eur}
for deg in degrees_eur:
    for N_val in Ns_eur:
        p_, _, _, _ = lsmc_bsde_european(p1, n=50, N=N_val, degree=deg)
        conv_eur[deg].append(p_)

fig1 = go.Figure()
colours = ['royalblue', 'darkorange', 'green']
for deg, col in zip(degrees_eur, colours):
    fig1.add_trace(go.Scatter(x=Ns_eur, y=conv_eur[deg], mode='lines+markers',
                              name=f'Hermite degree {deg}', line=dict(color=col)))
fig1.add_hline(y=bs_price, line=dict(color='black', dash='dash', width=2),
               annotation_text=f'Black-Scholes {bs_price:.4f}',
               annotation_position='top right')
fig1.update_layout(title='European call (BSDE-LSMC, Hermite basis)',
                   xaxis_title='N (paths)', yaxis_title='Call price estimate',
                   template='plotly_white', width=860, height=420)
fig1.show()

---
## Section 2 — Multi-Dimensional Basket Call

### Setup

$d$ correlated assets with **heterogeneous** $S_0^{(j)}$ and $\sigma_j$:

$$
dS_t^{(j)} = r\,S_t^{(j)}\,dt + \sigma_j\,S_t^{(j)}\,dW_t^{(j),\text{corr}},\qquad
d\langle W^{(i)},W^{(j)}\rangle_t = \rho_{ij}\,dt.
$$

Payoff: $\Phi(S_T)=\bigl(\frac{1}{d}\sum_j S_T^{(j)}-K\bigr)^+$.

### $Z$ Regression and Partial Delta Recovery

The BSDE is written w.r.t. **independent** BMs. The DP $Z$ regression:

$$
Z_{t_i,k}^{\text{ind}} = \frac{1}{\Delta t}\hat{\mathbb{E}}\!\left[Y_{t_{i+1}}\,\Delta W_{i,k}^{\text{ind}}\;\middle|\;X_{t_i}\right].
$$

By the chain rule of Ito's formula, $dV = \sum_j\Delta_j\,dS_j = \sum_k Z_k^{\text{ind}}\,dW_k^{\text{ind}}$ gives:

$$
Z_k^{\text{ind}} = \sum_j L_{jk}\,\sigma_j S_j\,\Delta_j
\quad\Longrightarrow\quad
Z^{\text{ind}} = L^\top\operatorname{diag}(\boldsymbol\sigma\odot\mathbf{S}_0)\,\boldsymbol\Delta
\quad\Longrightarrow\quad
\boldsymbol\Delta = \frac{(L^\top)^{-1}\,Z^{\text{ind}}}{\boldsymbol\sigma\odot\mathbf{S}_0}.
$$

Valid for any heterogeneous $\sigma_j$, $S_0^{(j)}$. Reduces to $\Delta_j=Z_j^{\text{ind}}/(\sigma_j S_j)$ when $\rho=I$.

### Regression State

- **$Y$:** scalar Hermite basis on $\log(A_t/K)$ where $A_t=\frac{1}{d}\sum_j S_t^{(j)}$ — sufficient statistic for the payoff.
- **$Z$:** full degree-2 polynomial in $\log(S_t^{(j)}/S_0^{(j)})$ — captures per-asset sensitivity without exploding basis size.

### Note on Delta Precision

The $Z$ estimator $\hat Z_k = N^{-1}\sum_j Y_1^{(j)}\Delta W_{0,k}^{\text{ind},(j)}/\Delta t$ has standard deviation $\approx \sigma_Y/\sqrt{N\Delta t}$. At $N=200\text{k}$, $n=50$, $\Delta t=0.02$ this gives approximately $\pm 5\%$ (1$\sigma$) per partial delta. Results are averaged over $R=8$ independent runs and reported with $\pm 2\hat\sigma$ Monte Carlo confidence intervals.


In [8]:
def mc_basket_antithetic(bp: BasketParams, N: int = 500_000, seed: int = 0):
    # Antithetic MC: price from +Z and -Z halves variance vs plain MC.
    # Returns (price, 95% CI half-width, partial_deltas via bump-reprice).
    np.random.seed(seed)
    d = len(bp.S0_vec)
    L = bp.cholesky()

    def _price(s0_vec: np.ndarray) -> tuple:
        Z     = np.random.randn(d, N)
        dW    = np.sqrt(bp.T) * (L @ Z)
        drift = (bp.r - 0.5 * bp.sigma_vec**2) * bp.T
        ST    = s0_vec[:, None] * np.exp(drift[:, None] + bp.sigma_vec[:, None] * dW)
        STa   = s0_vec[:, None] * np.exp(drift[:, None] - bp.sigma_vec[:, None] * dW)
        disc  = np.exp(-bp.r * bp.T)
        pv    = 0.5 * disc * (np.maximum(ST.mean(0) - bp.K, 0)
                              + np.maximum(STa.mean(0) - bp.K, 0))
        return pv.mean(), 1.96 * pv.std() / np.sqrt(N)

    price, ci = _price(bp.S0_vec)
    delta_vec = np.zeros(d)
    for j in range(d):
        h = 0.01 * bp.S0_vec[j]
        bu = bp.S0_vec.copy(); bu[j] += h
        bd = bp.S0_vec.copy(); bd[j] -= h
        delta_vec[j] = (_price(bu)[0] - _price(bd)[0]) / (2.0 * h)
    return price, ci, delta_vec

In [9]:
def lsmc_bsde_basket(bp: BasketParams, n: int, N: int, degree_Z: int = 2):
    # Basket call via BSDE-LSMC.
    # Y: scalar Hermite on log(avg(S)/K).
    # Z: degree-2 polynomial in log(S_j/S0_j) + dW_ind.
    # Delta: (L^T)^{-1} @ z_ind / (sigma * S0)
    d      = len(bp.S0_vec)
    dt     = bp.T / n
    disc   = 1.0 - bp.r * dt
    LT_inv = np.linalg.inv(bp.cholesky().T)

    S, dW_ind, _ = simulate_gbm_basket(bp, n, N)

    Y    = np.zeros((n + 1, N))
    Y[n] = np.maximum(S[n].mean(axis=1) - bp.K, 0.0)

    for i in range(n - 1, 0, -1):
        A_i  = S[i].mean(axis=1)
        B_Y  = hermite_basis_1d(np.log(np.clip(A_i / bp.K, 1e-8, None)), degree=4)
        Y[i] = disc * ols_predict(B_Y, Y[i + 1])

    log_s_rel = np.log(np.clip(S[1] / bp.S0_vec, 1e-8, None))
    B_Z0      = poly_basis_nd(log_s_rel, degree=degree_Z)
    z_ind     = np.array([ols_predict(B_Z0, Y[1] * dW_ind[0, :, k]).mean() / dt
                          for k in range(d)])

    price     = disc * Y[1].mean()
    delta_vec = LT_inv @ z_ind / (bp.sigma_vec * bp.S0_vec)
    return price, delta_vec, Y

In [10]:
# ── Market parameters ─────────────────────────────────────────────────────────
bp3 = BasketParams(
    S0_vec    = np.array([100.0, 105.0,  95.0]),
    sigma_vec = np.array([0.20,  0.25,   0.18]),
    K=100., r=0.05, rho=0.40, T=1.0)

bp5 = BasketParams(
    S0_vec    = np.array([100.0, 102.0,  98.0, 105.0,  95.0]),
    sigma_vec = np.array([0.20,  0.22,   0.25,  0.18,  0.23]),
    K=100., r=0.05, rho=0.35, T=1.0)

# ── MC benchmarks (fixed seed — deterministic) ────────────────────────────────
mc_p3, mc_ci3, mc_d3 = mc_basket_antithetic(bp3, N=500_000, seed=0)
mc_p5, mc_ci5, mc_d5 = mc_basket_antithetic(bp5, N=500_000, seed=0)
print(f"3-D MC: {mc_p3:.4f}  +/- {mc_ci3:.4f}")
print(f"5-D MC: {mc_p5:.4f}  +/- {mc_ci5:.4f}")

# ── BSDE-LSMC: R runs, each with N=200k paths (higher N for stable deltas) ────
# FIX 1: N=200k (was 50k) — reduces delta 1-sigma from ~9% to ~4.5%
# FIX 2: R=8 independent runs -> report mean +/- 2*std (95% MC CI on delta)
R = 8
N_BASK = 200_000

bsde_prices_3, bsde_deltas_3 = [], []
bsde_prices_5, bsde_deltas_5 = [], []

np.random.seed(SEED)   # FIX 3: local seed
for run in range(R):
    p3, d3, _ = lsmc_bsde_basket(bp3, n=50, N=N_BASK, degree_Z=2)
    p5, d5, _ = lsmc_bsde_basket(bp5, n=50, N=N_BASK, degree_Z=2)
    bsde_prices_3.append(p3); bsde_deltas_3.append(d3)
    bsde_prices_5.append(p5); bsde_deltas_5.append(d5)

bsde_p3_mean = np.mean(bsde_prices_3);  bsde_p3_std = np.std(bsde_prices_3)
bsde_p5_mean = np.mean(bsde_prices_5);  bsde_p5_std = np.std(bsde_prices_5)
bsde_d3_mean = np.mean(bsde_deltas_3, axis=0)
bsde_d3_std  = np.std(bsde_deltas_3,  axis=0)
bsde_d5_mean = np.mean(bsde_deltas_5, axis=0)
bsde_d5_std  = np.std(bsde_deltas_5,  axis=0)

print(f"\n3-D BSDE price: {bsde_p3_mean:.4f}  +/- {2*bsde_p3_std:.4f}  (2-sigma, R={R} runs x N={N_BASK:,})")
print(f"5-D BSDE price: {bsde_p5_mean:.4f}  +/- {2*bsde_p5_std:.4f}  (2-sigma, R={R} runs x N={N_BASK:,})")

3-D MC: 9.1251  +/- 0.0161
5-D MC: 8.5710  +/- 0.0142

3-D BSDE price: 9.1230  +/- 0.0490  (2-sigma, R=8 runs x N=200,000)
5-D BSDE price: 8.5884  +/- 0.0358  (2-sigma, R=8 runs x N=200,000)


In [11]:
for tag, bp, bsde_p, bsde_p_std, mc_p, mc_ci, bsde_d, bsde_d_std, mc_d in [
    ("3-D", bp3, bsde_p3_mean, bsde_p3_std, mc_p3, mc_ci3, bsde_d3_mean, bsde_d3_std, mc_d3),
    ("5-D", bp5, bsde_p5_mean, bsde_p5_std, mc_p5, mc_ci5, bsde_d5_mean, bsde_d5_std, mc_d5),
]:
    d = len(bp.S0_vec)
    print("=" * 70)
    print(f"{tag} Basket Call — Price")
    print(f"  MC antithetic   : {mc_p:.6f}  +/- {mc_ci:.6f}  (95% CI, N=500k antithetic)")
    print(f"  BSDE-LSMC       : {bsde_p:.6f}  +/- {2*bsde_p_std:.6f}  (2-sigma, {R} runs x N={N_BASK:,})")
    print(f"  Price bias      : {bsde_p - mc_p:+.6f}  ({(bsde_p-mc_p)/mc_p:+.4%})")
    print()
    print(f"{tag} Basket Call — Partial Deltas  [MC benchmark vs BSDE mean +/- 2-sigma CI]")
    labels = [f'S{j+1}  (S0={bp.S0_vec[j]:.0f}, sig={bp.sigma_vec[j]:.2f})' for j in range(d)]
    print(f"  {'Asset':<28} {'MC':>8}  {'BSDE':>8}  {'2-sigma':>8}  {'bias':>8}  {'bias %':>7}")
    print("  " + "-" * 68)
    for j in range(d):
        bias    = bsde_d[j] - mc_d[j]
        bias_pct= bias / abs(mc_d[j]) * 100
        print(f"  {labels[j]:<28} {mc_d[j]:>8.4f}  {bsde_d[j]:>8.4f}  "
              f"+/-{2*bsde_d_std[j]:>6.4f}  {bias:>+8.4f}  {bias_pct:>+6.1f}%")
    print()

3-D Basket Call — Price
  MC antithetic   : 9.125132  +/- 0.016121  (95% CI, N=500k antithetic)
  BSDE-LSMC       : 9.122979  +/- 0.049019  (2-sigma, 8 runs x N=200,000)
  Price bias      : -0.002153  (-0.0236%)

3-D Basket Call — Partial Deltas  [MC benchmark vs BSDE mean +/- 2-sigma CI]
  Asset                              MC      BSDE   2-sigma      bias   bias %
  --------------------------------------------------------------------
  S1  (S0=100, sig=0.20)         0.2231    0.2157  +/-0.0119   -0.0074    -3.3%
  S2  (S0=105, sig=0.25)         0.2253    0.2189  +/-0.0077   -0.0065    -2.9%
  S3  (S0=95, sig=0.18)          0.2120    0.2143  +/-0.0152   +0.0023    +1.1%

5-D Basket Call — Price
  MC antithetic   : 8.570991  +/- 0.014203  (95% CI, N=500k antithetic)
  BSDE-LSMC       : 8.588387  +/- 0.035789  (2-sigma, 8 runs x N=200,000)
  Price bias      : +0.017395  (+0.2030%)

5-D Basket Call — Partial Deltas  [MC benchmark vs BSDE mean +/- 2-sigma CI]
  Asset                      

**Reading the delta table.** The `2-sigma` column is the Monte Carlo standard deviation of the BSDE delta estimator over $R$ independent runs ($\approx 95\%$ confidence interval). A `bias` that is small relative to `2-sigma` is indistinguishable from zero — the algorithm is unbiased; variance is the limiting factor. A `bias` larger than `2-sigma` indicates a structural error.

At $N=200\text{k}$ per run the typical `2-sigma` is $\approx 4$–$6\%$ of the true delta. Running more paths ($N\to\infty$) or more replicates ($R\to\infty$) shrinks this band as $N^{-1/2}$.


In [12]:
# ── Convergence: 3-D basket price ─────────────────────────────────────────────
Ns_bask = list(range(10_000, 210_000, 20_000))
conv_bask3 = []
np.random.seed(SEED)
for N_val in Ns_bask:
    p_, _, _ = lsmc_bsde_basket(bp3, n=30, N=N_val, degree_Z=2)
    conv_bask3.append(p_)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=Ns_bask, y=conv_bask3, mode='lines+markers',
                          name='BSDE-LSMC price', line=dict(color='royalblue')))
fig2.add_hline(y=mc_p3, line=dict(color='black', dash='dash', width=2),
               annotation_text=f'MC benchmark  {mc_p3:.4f}',
               annotation_position='top right')
fig2.update_layout(title='3-D basket call (BSDE-LSMC)',
                   xaxis_title='N (paths)', yaxis_title='Basket call price',
                   template='plotly_white', width=860, height=420)
fig2.show()

---
## Section 3 — 1-D American Put via Reflected BSDE

### RBSDE and Optimal Stopping

El Karoui, Peng & Quenez (1997): the American put price solves the **Reflected BSDE**

$$
-dY_t = -rY_t\,dt - Z_t\,dW_t^{\text{ind}} - dK_t,\quad
Y_T=(K-S_T)^+,\quad Y_t\ge h_t:=(K-S_t)^+,
$$

where $K_t$ is an increasing process (Skorokhod reflection) satisfying $\int_0^T(Y_t-h_t)\,dK_t=0$.

### Theorem: Discretised RBSDE $\equiv$ Longstaff–Schwartz

**Proof.** At each step $i$ compute the unconstrained continuation:

$$
\tilde Y_{t_i} = (1-r\Delta t)\,\hat{\mathbb{E}}[Y_{t_{i+1}}\mid S_{t_i}],
$$

then enforce the obstacle:

$$
Y_{t_i} = \max\!\bigl(\tilde Y_{t_i},\;h(t_i,S_{t_i})\bigr).
$$

This is the Snell envelope recursion (smallest supermartingale dominating $h$). The discrete reflection increments $\Delta K_{t_i}=\max(h_{t_i}-\tilde Y_{t_i},0)\ge 0$ satisfy the Skorokhod condition. In Longstaff–Schwartz (2001): regress discounted future cashflows, set $Y_{t_i}=\max(\hat C,h_{t_i})$. Algebraically identical. $\blacksquare$

### ITM-Only Regression

Fit the continuation regression **only on ITM paths** ($h(t_i,S_{t_i})>0$), apply to all paths. OTM paths default to `disc * Y[i+1]` (hold). Reasons:

- Exercise decision only relevant where $h>0$.
- Hermite basis fit on $\log(S/K)<0$ extrapolated to $\log(S/K)>0$ produces noisy continuation estimates near the boundary, biasing price upward by $\sim10$–$15\%$ when all paths are included.

### $Z$ and Delta

$Z$ is regressed on **all** paths using the **reflected** $Y$ (which encodes the exercise decision). This gives the American delta, not the European delta. Using the terminal payoff directly would give the European put delta — $\approx9\%$ wrong near ATM.


In [13]:
def lsmc_bsde_american(p: MarketParams1D, n: int, N: int, degree: int = 4):
    # American put — Reflected BSDE-LSMC (LS 2001 + GLW 2005).
    # Y: ITM-only regression + hold default for OTM.
    # Z: all-paths martingale regression on reflected Y.
    dt   = p.T / n
    disc = 1.0 - p.r * dt
    S, dW = simulate_gbm_1d(p, n, N)

    intrinsic = np.maximum(p.K - S, 0.0)
    Y = np.zeros((n + 1, N))
    Z = np.zeros((n + 1, N))
    Y[n] = intrinsic[n]

    for i in range(n - 1, 0, -1):
        Y[i] = disc * Y[i + 1]                # default: hold
        itm  = intrinsic[i] > 0
        if itm.sum() > degree + 2:
            x_itm = np.log(np.clip(S[i, itm], 1e-8, None) / p.K)
            B_itm = hermite_basis_1d(x_itm, degree)
            cont  = ols_predict(B_itm, disc * Y[i + 1, itm])
            Y[i, itm] = np.where(intrinsic[i, itm] > cont, intrinsic[i, itm], cont)
        B_all = hermite_basis_1d(np.log(np.clip(S[i], 1e-8, None) / p.K), degree)
        Z[i]  = ols_predict(B_all, Y[i + 1] * dW[i]) / dt

    B0   = hermite_basis_1d(np.log(np.clip(S[1], 1e-8, None) / p.K), degree)
    Z[0] = ols_predict(B0, Y[1] * dW[0]) / dt
    price = disc * Y[1].mean()
    delta = Z[0].mean() / (p.sigma * p.S0)
    return price, delta, Y, Z


def crr_american_put(p: MarketParams1D, M: int = 1000):
    dt   = p.T / M
    u    = np.exp(p.sigma * np.sqrt(dt))
    d_   = 1.0 / u
    disc = np.exp(-p.r * dt)
    p_up = (np.exp(p.r * dt) - d_) / (u - d_)
    j    = np.arange(M + 1)
    V    = np.maximum(p.K - p.S0 * (u ** (M - 2 * j)), 0.0)
    for step in range(M - 1, -1, -1):
        Ss = p.S0 * (u ** (step - 2 * np.arange(step + 1)))
        V  = disc * (p_up * V[:step + 1] + (1 - p_up) * V[1:step + 2])
        V  = np.maximum(V, p.K - Ss)
    price = float(V[0])
    V2 = np.maximum(p.K - p.S0 * (u ** (M - 2 * np.arange(M + 1))), 0.0)
    for step in range(M - 1, 0, -1):
        Ss = p.S0 * (u ** (step - 2 * np.arange(step + 1)))
        V2 = disc * (p_up * V2[:step + 1] + (1 - p_up) * V2[1:step + 2])
        V2 = np.maximum(V2, p.K - Ss)
    delta = float((V2[0] - V2[1]) / (p.S0 * u - p.S0 * d_))
    return price, delta

In [14]:
p_am = MarketParams1D(S0=100., K=100., r=0.05, sigma=0.20, T=1.0)

crr_price, crr_delta = crr_american_put(p_am, M=1000)
crr_z0 = p_am.sigma * p_am.S0 * crr_delta

# R independent runs at N=100k — mean price and delta with 2-sigma CI
R_AM = 10
np.random.seed(SEED)   # FIX 3: local seed
am_prices, am_deltas = [], []
for _ in range(R_AM):
    p_, d_, _, _ = lsmc_bsde_american(p_am, n=50, N=100_000, degree=4)
    am_prices.append(p_); am_deltas.append(d_)

am_price_mean = np.mean(am_prices);   am_price_std = np.std(am_prices)
am_delta_mean = np.mean(am_deltas);   am_delta_std = np.std(am_deltas)
am_z0_mean    = am_delta_mean * p_am.sigma * p_am.S0
am_z0_std     = am_delta_std  * p_am.sigma * p_am.S0

print(f"{'Method':<30} {'Price':>10}  {'Delta':>9}  {'Z0':>10}")
print("-" * 65)
print(f"{'CRR tree (M=1000)':<30} {crr_price:>10.6f}  {crr_delta:>9.6f}  {crr_z0:>10.6f}")
print(f"{'BSDE-LSMC mean':<30} {am_price_mean:>10.6f}  {am_delta_mean:>9.6f}  {am_z0_mean:>10.6f}")
print(f"{'BSDE-LSMC 2-sigma CI':<30} {2*am_price_std:>10.6f}  {2*am_delta_std:>9.6f}  {2*am_z0_std:>10.6f}")
print("-" * 65)
print(f"{'Bias (mean - CRR)':<30} {am_price_mean-crr_price:>+10.6f}  {am_delta_mean-crr_delta:>+9.6f}  {am_z0_mean-crr_z0:>+10.6f}")
print(f"{'Bias / CRR':<30} {(am_price_mean-crr_price)/crr_price:>+9.4%}  {(am_delta_mean-crr_delta)/abs(crr_delta):>+8.4%}  {(am_z0_mean-crr_z0)/abs(crr_z0):>+9.4%}")
print(f"{'Bias / 2-sigma':<30} {abs(am_price_mean-crr_price)/(2*am_price_std):>10.2f}x  {abs(am_delta_mean-crr_delta)/(2*am_delta_std):>9.2f}x")
print()
print(f"R={R_AM} independent runs x N=100,000 paths each.")
print("'Bias / 2-sigma' < 1 means the deviation is within noise — no structural error.")

Method                              Price      Delta          Z0
-----------------------------------------------------------------
CRR tree (M=1000)                6.089595  -0.411114   -8.222284
BSDE-LSMC mean                   6.097927  -0.414592   -8.291841
BSDE-LSMC 2-sigma CI             0.039074   0.016747    0.334937
-----------------------------------------------------------------
Bias (mean - CRR)               +0.008332  -0.003478   -0.069556
Bias / CRR                      +0.1368%  -0.8459%   -0.8459%
Bias / 2-sigma                       0.21x       0.21x

R=10 independent runs x N=100,000 paths each.
'Bias / 2-sigma' < 1 means the deviation is within noise — no structural error.


### Verification Across Moneyness

Each row uses a fresh local seed — results are independent of each other and of prior cells.


In [15]:
spots = [85., 90., 95., 100., 105., 110., 115.]
rows  = []
np.random.seed(SEED + 1)   # local seed for moneyness sweep
for s0 in spots:
    p_s = MarketParams1D(S0=s0, K=100., r=0.05, sigma=0.20, T=1.0)
    cp, cd = crr_american_put(p_s, M=1000)
    # 5 runs per spot to get a CI
    ps, ds = [], []
    for _ in range(5):
        lp, ld, _, _ = lsmc_bsde_american(p_s, n=50, N=80_000, degree=4)
        ps.append(lp); ds.append(ld)
    lp_m, lp_s = np.mean(ps), np.std(ps)
    ld_m, ld_s = np.mean(ds), np.std(ds)
    rows.append({
        'S0'          : s0,
        'CRR price'   : cp,
        'BSDE price'  : lp_m,
        'Price +/-2s' : 2 * lp_s,
        'Price bias%' : (lp_m - cp) / cp * 100,
        'CRR delta'   : cd,
        'BSDE delta'  : ld_m,
        'Delta +/-2s' : 2 * ld_s,
        'Delta bias%' : (ld_m - cd) / abs(cd) * 100,
    })
df_am = pd.DataFrame(rows).set_index('S0')
print(df_am.to_string(float_format='{:.4f}'.format))

       CRR price  BSDE price  Price +/-2s  Price bias%  CRR delta  BSDE delta  Delta +/-2s  Delta bias%
S0                                                                                                     
85.0     15.3155     15.3010       0.0239      -0.0947    -0.8492     -0.8515       0.0468      -0.2790
90.0     11.4934     11.4988       0.0337       0.0474    -0.6833     -0.6684       0.0232       2.1783
95.0      8.4507      8.4513       0.0100       0.0071    -0.5368     -0.5326       0.0231       0.7937
100.0     6.0896      6.1025       0.0312       0.2127    -0.4111     -0.4118       0.0088      -0.1646
105.0     4.3048      4.3086       0.0327       0.0875    -0.3069     -0.3078       0.0048      -0.3017
110.0     2.9879      2.9858       0.0251      -0.0683    -0.2236     -0.2190       0.0076       2.0606
115.0     2.0361      2.0369       0.0254       0.0375    -0.1592     -0.1605       0.0026      -0.7954


In [16]:
# ── Convergence: American put price ──────────────────────────────────────────
Ns_am = list(range(10_000, 110_000, 10_000))
conv_am_d3, conv_am_d4 = [], []
np.random.seed(SEED)
for N_val in Ns_am:
    p3_, _, _, _ = lsmc_bsde_american(p_am, n=50, N=N_val, degree=3)
    p4_, _, _, _ = lsmc_bsde_american(p_am, n=50, N=N_val, degree=4)
    conv_am_d3.append(p3_); conv_am_d4.append(p4_)

fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=Ns_am, y=conv_am_d3, mode='lines+markers',
                          name='Hermite degree 3', line=dict(color='royalblue')))
fig3.add_trace(go.Scatter(x=Ns_am, y=conv_am_d4, mode='lines+markers',
                          name='Hermite degree 4', line=dict(color='darkorange')))
fig3.add_hline(y=crr_price, line=dict(color='black', dash='dash', width=2),
               annotation_text=f'CRR tree  {crr_price:.4f}',
               annotation_position='top right')
fig3.update_layout(title='American put (Reflected BSDE-LSMC)',
                   xaxis_title='N (paths)', yaxis_title='Put price',
                   template='plotly_white', width=860, height=420)
fig3.show()